# Stochastic Calculus from Zero · Episode 5: Conditional expectation: the best guess

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The coin world.** A share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses, eight possible worlds. Heads has a 60% real-world chance; a deposit grows 2% a step. Made up for teaching.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. Guessing with no information · 4. Guessing block by block · 5. The tower property · 6. Taking out what is known · 7. Best in what sense? · 8. This episode's question · 9. Exercises · 10. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 5,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)

## 2. Last episode's answer

Four tosses: $2^4 = 16$ histories, but only 5 different final prices (0 to 4 heads).

In [ ]:
out["prev"] = {"histories": 16, "prices": 5}

## 3. Guessing with no information

Before any toss, the best single guess of $S_3$ (in the least-squares sense, Section 7) is its plain expectation, $E_P[S_3] = 100 \times 1.04^3$.

In [ ]:
cw = CoinWorld()
S3 = lambda w: cw.price(w, 3)
e0 = cw.expect(S3)
out["e0"] = e0
e0

## 4. Guessing block by block

After some tosses you know which block you are in, so you average $S_3$ **within that block** only, using the weights of its worlds. That is the conditional expectation $E_P[S_3 \mid \mathcal{F}_n]$: one guess per block, so a random variable, not a single number. Each remaining toss multiplies the price by 1.04 on average, so $E_P[S_3 \mid \mathcal{F}_n] = S_n \times 1.04^{3-n}$.

In [ ]:
g1, g2 = cw.given(1, S3), cw.given(2, S3)
for b, v in g1.items():
    assert abs(v - cw.price(b, 1) * 1.04 ** 2) < 1e-9
for b, v in g2.items():
    assert abs(v - cw.price(b, 2) * 1.04) < 1e-9
call = lambda w: max(cw.price(w, 3) - 100, 0)
c2 = cw.given(2, call)
rows = cw.table(g1=lambda w: g1[w[:1]], g2=lambda w: g2[w[:2]], call=call, c2=lambda w: c2[w[:2]])
out["coin"] = {"worlds": rows}
out["call"] = {"strike": 100, "F2": c2, "E0": cw.expect(call)}
out["given"] = {"F1": g1, "F2": g2, "growth": 1.04}
pd.DataFrame(rows)[["world", "P", "S1", "S2", "S3", "g1", "g2"]]

## 5. The tower property

Average the time-1 guesses with the time-1 weights and you get back the time-0 guess: $E\big[E[S_3 \mid \mathcal{F}_1]\big] = E[S_3]$. More generally, a guess of a later guess, made with less information, is just the earlier guess: $E\big[E[X \mid \mathcal{F}_2] \mid \mathcal{F}_1\big] = E[X \mid \mathcal{F}_1]$.

In [ ]:
tower0 = cw.p * g1["H"] + (1 - cw.p) * g1["T"]
tower1 = {b: cw.p * g2[b + "H"] + (1 - cw.p) * g2[b + "T"] for b in "HT"}
assert abs(tower0 - e0) < 1e-9 and all(abs(tower1[b] - g1[b]) < 1e-9 for b in "HT")
out["tower"] = {"p": cw.p, "q_weight": 1 - cw.p, "H": g1["H"], "T": g1["T"], "result": tower0}
out["tower"]

## 6. Taking out what is known

At time 1 the price $S_1$ is known, so inside a conditional expectation given $\mathcal{F}_1$ it behaves like a constant: $E[S_1 S_3 \mid \mathcal{F}_1] = S_1\, E[S_3 \mid \mathcal{F}_1]$.

In [ ]:
lhs = cw.given(1, lambda w: cw.price(w, 1) * S3(w))
rhs = {b: cw.price(b, 1) * g1[b] for b in "HT"}
assert all(abs(lhs[b] - rhs[b]) < 1e-9 for b in "HT")
out["take_out"] = {"S1_H": cw.price("H", 1), "lhs_H": lhs["H"], "rhs_H": rhs["H"]}
out["take_out"]

## 7. Best in what sense?

Simulate 100,000 worlds with the real-world odds. Fit $S_3$ by least squares using only what is known after one toss (an indicator for "first toss heads" and one for "first toss tails"). The fitted values are the block averages, up to sampling noise: **conditional expectation is the least-squares projection onto what you know**. No other guess that uses only $\mathcal{F}_1$ has a smaller average squared error.

In [ ]:
rng = np.random.default_rng(seed)
heads = rng.random((100_000, 3)) < cw.p
s3 = cw.s0 * np.where(heads, cw.up, cw.down).prod(axis=1)
X = np.column_stack([heads[:, 0], ~heads[:, 0]]).astype(float)
beta = np.linalg.lstsq(X, s3, rcond=None)[0]
mse_best = float(((s3 - X @ beta) ** 2).mean())
mse_naive = float(((s3 - s3.mean()) ** 2).mean())
assert abs(beta[0] - g1["H"]) < 0.5 and abs(beta[1] - g1["T"]) < 0.5 and mse_best < mse_naive
out["fit"] = {"n": 100_000, "H": float(beta[0]), "T": float(beta[1]), "rmse_best": math.sqrt(mse_best), "rmse_naive": math.sqrt(mse_naive),
              "table": [{"block": "First toss heads", "average": g1["H"], "fit": float(beta[0])},
                        {"block": "First toss tails", "average": g1["T"], "fit": float(beta[1])}]}
out["fit"]["table"], out["fit"]["rmse_best"], out["fit"]["rmse_naive"]

## 8. This episode's question

If heads had a 50% chance, each toss would multiply the price by $0.5 \times 1.2 + 0.5 \times 0.8 = 1$ on average, so after a first head the best guess of $S_3$ would be $S_1 = 120$.

In [ ]:
fair = CoinWorld(p=0.5)
out["question"] = {"p": 0.5, "answer": fair.given(1, lambda w: fair.price(w, 3))["H"]}
assert abs(out["question"]["answer"] - 120) < 1e-9

## 9. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Find $E_P[\text{heads} \mid \mathcal{F}_1]$ in both blocks.

<details><summary>Answer</summary>

After H: 1 head so far plus $2 \times 0.6 = 1.2$ expected, so 2.2. After T: $0 + 1.2 = 1.2$.

</details>

**Exercise 2 (By hand).** Show that $E_P[S_2 \mid \mathcal{F}_1] = 1.04\, S_1$ in both blocks.

<details><summary>Answer</summary>

After H: $0.6 \times 144 + 0.4 \times 96 = 86.4 + 38.4 = 124.8 = 1.04 \times 120$. After T: $0.6 \times 96 + 0.4 \times 64 = 57.6 + 25.6 = 83.2 = 1.04 \times 80$.

</details>

**Exercise 3 (By hand).** Check the tower property at time 2 for the block H: average $E[S_3 \mid \mathcal{F}_2]$ over HH and HT.

<details><summary>Answer</summary>

$0.6 \times 149.76 + 0.4 \times 99.84 = 89.856 + 39.936 = 129.792 = E[S_3 \mid \mathcal{F}_1]$ on H. ✓

</details>

**Exercise 4 (In code).** Compute $E_P[\text{call} \mid \mathcal{F}_2]$ for the call struck at 100.

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
call = lambda w: max(cw.price(w, 3) - 100, 0)
gc = cw.given(2, call)
assert abs(gc["HH"] - (0.6 * 72.8 + 0.4 * 15.2)) < 1e-9 and gc["TT"] == 0
print({b: round(v, 4) for b, v in gc.items()})

**Exercise 5 (In code).** Repeat Section 7's least-squares fit with indicators for the four time-2 blocks. Do the fitted values match $E[S_3 \mid \mathcal{F}_2]$?

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
key = heads[:, 0] * 2 + heads[:, 1]            # 3 = HH, 2 = HT, 1 = TH, 0 = TT
X2 = np.column_stack([key == k for k in (3, 2, 1, 0)]).astype(float)
b2 = np.linalg.lstsq(X2, s3, rcond=None)[0]
for b, v in zip(("HH", "HT", "TH", "TT"), b2):
    assert abs(v - g2[b]) < 0.6
print({k: round(float(v), 2) for k, v in zip(("HH", "HT", "TH", "TT"), b2)})

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")